# Lab 04 — Spark SQL dan Data Analytics
**SQL/DataFrame, built-ins, aggregation, window, KPI, execution plan**

Notebook ini dirancang untuk environment pelatihan: Spark 3.5.7 standalone, Jupyter, 2 worker, masing-masing 2 core / 2 GB.  
**Prinsip lab:** prediksi → jalankan → amati → jelaskan. Jangan hanya mengejar output akhir.

## Capaian praktik
Menggunakan SQL dan DataFrame API sebagai dua pintu ke engine yang sama; membangun agregasi/KPI/window analysis; membaca execution plan dan menghubungkannya ke pertanyaan bisnis.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
import time, json, os, shutil

spark = (
    SparkSession.builder
    .appName("spark-training-lab")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("App name      :", spark.sparkContext.appName)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

## Dataset analitik sintetis
Dataset dibuat dengan Spark agar cukup besar untuk melihat plan/shuffle tetapi tetap aman untuk laptop lab.

In [ ]:
transaction_schema = T.StructType([
    T.StructField("trx_id", T.StringType(), False),
    T.StructField("taxpayer_id", T.StringType(), False),
    T.StructField("trx_date", T.DateType(), False),
    T.StructField("office", T.StringType(), False),
    T.StructField("tax_type", T.StringType(), False),
    T.StructField("channel", T.StringType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("amount", T.DoubleType(), False),
])

transactions_path = "file:///home/spark/datasets/transactions.csv"
transactions = (
    spark.read.schema(transaction_schema)
    .option("header", True)
    .csv(transactions_path)
)
transactions.createOrReplaceTempView("transactions")

transactions.show(5, truncate=False)
transactions.printSchema()
print("rows:", transactions.count())
transactions.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["trx_id", "taxpayer_id", "status", "amount"]
]).show()

## Lab 1 — Eksplorasi dengan Spark SQL (30 menit)
**Rujukan slide:** Deck 04, slide 57.

Validasi schema, jumlah baris, null, dan tipe `amount`; lalu tulis analisis yang sama dengan SQL dan DataFrame API.

In [ ]:
sql_result = spark.sql("""
SELECT office, amount, amount / 1000000.0 AS amount_million
FROM transactions
WHERE status = 'VALID' AND amount >= 1000000
ORDER BY amount DESC
LIMIT 20
""")

api_result = (
    transactions
    .filter((F.col("status") == "VALID") & (F.col("amount") >= 1_000_000))
    .select("office", "amount", (F.col("amount") / 1_000_000).alias("amount_million"))
    .orderBy(F.col("amount").desc())
    .limit(20)
)

sql_result.show(5)
api_result.show(5)
print("filtered rows:", transactions.filter((F.col("status") == "VALID") & (F.col("amount") >= 1_000_000)).count())

In [ ]:
print("--- SQL PLAN ---")
sql_result.explain("formatted")
print("--- DATAFRAME PLAN ---")
api_result.explain("formatted")

**Diskusi:** apakah SQL otomatis lebih cepat daripada DataFrame API? Bandingkan physical plan, bukan selera sintaks.

## Built-in function warm-up — NULL, date, dan CASE
Bagian ini mempraktikkan built-in functions pada Deck 04 slides 17–23 sebelum Lab 2.

In [ ]:
# Tambahkan beberapa NULL agar perilakunya terlihat.
with_null = transactions.withColumn(
    "amount2", F.when((F.xxhash64("trx_id") % 17) == 0, F.lit(None).cast("double")).otherwise(F.col("amount"))
)
with_null.createOrReplaceTempView("transactions_null")

spark.sql("""
SELECT
  COUNT(*) AS all_rows,
  COUNT(amount2) AS amount_not_null,
  SUM(CASE WHEN amount2 IS NULL THEN 1 ELSE 0 END) AS null_amount
FROM transactions_null
""").show()

In [ ]:
spark.sql("""
SELECT
  trx_id,
  trx_date,
  YEAR(trx_date) AS yr,
  MONTH(trx_date) AS mn,
  office,
  amount,
  CASE
    WHEN amount >= 10000000 THEN 'HIGH'
    WHEN amount >= 1000000 THEN 'MEDIUM'
    ELSE 'LOW'
  END AS amount_segment
FROM transactions
LIMIT 10
""").show(truncate=False)

## Lab 2 — Aggregation dan KPI Dasar (35 menit)
**Rujukan slide:** Deck 04, slide 58.

Bangun KPI pada grain `office × tax_type`, lalu bandingkan exact dan approximate distinct count.

In [ ]:
kpi = spark.sql("""
SELECT
  office,
  tax_type,
  COUNT(*) AS trx_count,
  COUNT(DISTINCT taxpayer_id) AS taxpayer_count,
  SUM(amount) AS total_amount,
  AVG(amount) AS avg_amount
FROM transactions
WHERE status = 'VALID'
GROUP BY office, tax_type
""")
kpi.show(truncate=False)

spark.sql("""
SELECT office, tax_type, COUNT(*) AS trx_count
FROM transactions
WHERE status = 'VALID'
GROUP BY office, tax_type
HAVING COUNT(*) >= 1000
ORDER BY trx_count DESC
""").show(truncate=False)

transactions.filter(F.col("status") == "VALID").groupBy("office", "tax_type").agg(
    F.countDistinct("taxpayer_id").alias("exact_taxpayer_count"),
    F.approx_count_distinct("taxpayer_id").alias("approx_taxpayer_count"),
).show(truncate=False)

## Lab 3 — Aggregation + Window: ranking dan growth (35–40 menit)

**Rujukan slide:** Deck 04, slide 59.

In [ ]:
from pyspark.sql.window import Window

rank_w = Window.partitionBy("office").orderBy(F.col("amount").desc())
ranked = (
    transactions.filter(F.col("status") == "VALID")
    .withColumn("row_number", F.row_number().over(rank_w))
    .withColumn("rank", F.rank().over(rank_w))
    .withColumn("dense_rank", F.dense_rank().over(rank_w))
)
ranked.filter(F.col("row_number") <= 3).orderBy("office", "row_number").show(20, truncate=False)

In [ ]:
monthly = (
    transactions.filter(F.col("status") == "VALID")
    .withColumn("month", F.date_trunc("month", "trx_date"))
    .groupBy("office", "month")
    .agg(F.sum("amount").alias("total_amount"))
)

w = Window.partitionBy("office").orderBy("month")
monthly_growth = (
    monthly
    .withColumn("prev_amount", F.lag("total_amount").over(w))
    .withColumn("delta", F.col("total_amount") - F.col("prev_amount"))
    .withColumn("growth_pct", F.when(F.col("prev_amount") != 0, F.col("delta") / F.col("prev_amount") * 100))
)
monthly_growth.orderBy("office", "month").show(30, truncate=False)

## Lab 4 — Studi kasus analitik bisnis (40–50 menit)
Jawab tiga pertanyaan dan tulis insight dalam bahasa bisnis, bukan hanya tabel.

**Rujukan slide:** Deck 04, slide 60.

In [ ]:
# Q1: office terbesar + count + avg
q1 = (
    transactions.filter(F.col("status") == "VALID")
    .groupBy("office")
    .agg(
        F.count("*").alias("trx_count"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("avg_amount")
    )
    .orderBy(F.col("total_amount").desc())
)
q1.show(truncate=False)

In [ ]:
# Q2: kontribusi channel per office
channel = (
    transactions.filter(F.col("status") == "VALID")
    .groupBy("office", "channel")
    .agg(F.sum("amount").alias("channel_amount"))
)
wo = Window.partitionBy("office")
wr = Window.partitionBy("office").orderBy(F.col("channel_amount").desc())
channel_share = (
    channel
    .withColumn("office_amount", F.sum("channel_amount").over(wo))
    .withColumn("share", F.col("channel_amount") / F.col("office_amount"))
    .withColumn("channel_rank", F.dense_rank().over(wr))
)
channel_share.orderBy("office", "channel_rank").show(30, truncate=False)

In [ ]:
# Q3: monthly trend + moving average 3 periode
w3 = Window.partitionBy("office").orderBy("month").rowsBetween(-2, 0)
trend = monthly_growth.withColumn("moving_avg_3", F.avg("total_amount").over(w3))
trend.orderBy("office", "month").show(30, truncate=False)

**Deliverable:** tulis 3 insight dan 1 keterbatasan analisis. Jangan menyimpulkan kausalitas dari pola agregat.

## Lab 5 — Membaca plan & menemukan bottleneck (30 menit)

**Rujukan slide:** Deck 04, slide 61.

In [ ]:
q = (
    transactions
    .filter(F.col("status") == "VALID")
    .groupBy("office", "tax_type")
    .agg(F.sum("amount").alias("total_amount"))
    .orderBy(F.col("total_amount").desc())
)
q.explain("formatted")
q.show(truncate=False)

**Checklist:**
1. Pastikan ada tepat satu `Scan csv`; filter dan kolom apa yang terlihat?
2. Di mana `Exchange` muncul dan operator apa yang memicunya?
3. Cocokkan SQL tab dengan Stages di Spark UI.
4. Apakah semua Exchange adalah masalah? Jelaskan.

## Mini Project — Dari data ke insight (30–40 menit)
Pilih satu:
- **Performance:** office mana memiliki volume/nilai tertinggi dan bagaimana tren 3 bulannya?
- **Channel:** channel dominan per office dan office mana yang polanya berbeda?
- **Distribution:** bandingkan mean, median/p50, p90, p99 dan identifikasi outlier.

**Deliverable:** 1 query utama + 1 `explain()` + 3 insight + 1 keterbatasan.

**Rujukan slide:** Deck 04, slide 62.